In [5]:
!pip install scikit-learn==1.9.1

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
df=pd.read_csv("/content/drive/MyDrive/ai_job_dataset.csv")

In [8]:
df.head()

,job_id,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,education_required,years_experience,industry,posting_date,application_deadline,job_description_length,benefits_score,company_name
0,AI00001,AI Research Scientist,90376,USD,SE,CT,China,M,China,50,"Tableau, PyTorch, Kubernetes, Linux, NLP",Bachelor,9,Automotive,2024-10-18,2024-11-07,1076,5.9,Smart Analytics
1,AI00002,AI Software Engineer,61895,USD,EN,CT,Canada,M,Ireland,100,"Deep Learning, AWS, Mathematics, Python, Docker",Master,1,Media,2024-11-20,2025-01-11,1268,5.2,TechCorp Inc
2,AI00003,AI Specialist,152626,USD,MI,FL,Switzerland,L,South Korea,0,"Kubernetes, Deep Learning, Java, Hadoop, NLP",Associate,2,Education,2025-03-18,2025-04-07,1974,9.4,Autonomous Tech
3,AI00004,NLP Engineer,80215,USD,SE,FL,India,M,India,50,"Scala, SQL, Linux, Python",PhD,7,Consulting,2024-12-23,2025-02-24,1345,8.6,Future Systems
4,AI00005,AI Consultant,54624,EUR,EN,PT,France,S,Singapore,100,"MLOps, Java, Tableau, Python",Master,0,Media,2025-04-15,2025-06-23,1989,6.6,Advanced Robotics


#Remove Outlier

In [9]:
Q1 = df['salary_usd'].quantile(0.25)
Q3 = df['salary_usd'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Lower bound: -44163.375
Upper bound: 260751.625


In [10]:
df = df[
    (df['salary_usd'] >= lower_bound) &
    (df['salary_usd'] <= upper_bound)
]

In [11]:
X = df.drop('salary_usd', axis=1)
y = df['salary_usd']

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

In [13]:
#Separate categorical and numerical variables.
numerical_features=df.select_dtypes(include=np.number).columns
categorical_features=df.select_dtypes(include='object').columns
print(numerical_features)
print(categorical_features)

Index(['salary_usd', 'remote_ratio', 'years_experience',
       'job_description_length', 'benefits_score'],
      dtype='object')
Index(['job_id', 'job_title', 'salary_currency', 'experience_level',
       'employment_type', 'company_location', 'company_size',
       'employee_residence', 'required_skills', 'education_required',
       'industry', 'posting_date', 'application_deadline', 'company_name'],
      dtype='object')


In [14]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [15]:
#ordinal columns
ordinal_features=[
    'experience_level',
    'company_size',
    'education_required'
]

In [16]:
ordinal_categories = [
    ['EN', 'MI', 'SE', 'EX'],
    ['Small', 'Medium', 'Large'],
     ['Bachelor','Master','Associate','PhD']
]

In [17]:
#categorical columns
categorical_features = [
    'job_title',
    'employment_type',
    'company_location',
    'company_name',
    'employee_residence',
    'industry',
    'salary_currency',
]

In [18]:
#numerical columns
numerical_features = [
    'years_experience',
    'job_description_length',
    'benefits_score',
    'remote_ratio'
]

In [19]:
#column transfer
preprocessor=ColumnTransformer(
    transformers=[
        (
           'ordinal',
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown='use_encoded_value',
                unknown_value=-1

            ),
            ordinal_features
        ),
          (
              'cat',
              OneHotEncoder(
                  handle_unknown='ignore',
              ),
              categorical_features
          ),
        (
           'num',
            'passthrough',
           numerical_features
        )
    ]
)

In [20]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

#Gradient Boosting

In [21]:
from sklearn.ensemble import GradientBoostingRegressor
model_gb=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        (
            'model',
            GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=4,
                random_state=42
            )
        )
    ]
)
model_gb.fit(X_train,y_train)
pred_gb=model_gb.predict(X_test)

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print(
    "MAE:",
    mean_absolute_error(y_test, pred_gb)
)

print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_test, pred_gb))
)

print(
    "R2:",
    r2_score(y_test, pred_gb)
)

MAE: 15209.714624281385
RMSE: 20389.89824837888
R2: 0.8374678331296801


#GridSearchCV

In [23]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

gbr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [2, 3, 5],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 5]
}

grid_search = GridSearchCV(
    estimator=gbr_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know mor

#Find best parameters

In [24]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV RMSE:")
print(-grid_search.best_score_)

Best Parameters:
{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2, 'model__n_estimators': 200}

Best CV RMSE:
20295.674124669826


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_gbr = grid_search.best_estimator_

y_pred_gbr = best_gbr.predict(X_test)

mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = mean_squared_error(y_test, y_pred_gbr) ** 0.5
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Results")
print("-------------------------")
print("MAE :", mae_gbr)
print("RMSE:", rmse_gbr)
print("R²  :", r2_gbr)

Gradient Boosting Results
-------------------------
MAE : 15174.140031444207
RMSE: 20291.05970132785
R²  : 0.8390397397098813


In [26]:
import joblib

joblib.dump(best_gbr, "salary_prediction_model.pkl")

['salary_prediction_model.pkl']

# Random Forest

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
model_rf=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        (
            'model',
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)
model_rf.fit(X_train,y_train)
pred_rf=model_rf.predict(X_test)

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print(
    "MAE:",
    mean_absolute_error(y_test, pred_rf)
)

print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_test, pred_rf))
)

print(
    "R2:",
    r2_score(y_test, pred_rf)
)

MAE: 15565.479863406796
RMSE: 20949.00047502109
R2: 0.8284321845244289


#Decision Tree


In [29]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
model_dt= Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',DecisionTreeRegressor(
            random_state=42
        )
        )
    ]
)
model_dt.fit(X_train,y_train)
y_train_pred=model_dt.predict(X_train)
y_pred=model_dt.predict(X_test)

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
mae=mean_absolute_error(y_test,y_pred)
rmse=np.sqrt(mean_squared_error(y_test,y_pred))
r2_train=r2_score(y_train,y_train_pred)
r2=r2_score(y_test,y_pred)
print("MAE:", mae)
print("RMSE:", rmse)
print(r2_train)
print("R² Score:", r2)

MAE: 20848.70523415978
RMSE: 28407.61869450803
1.0
R² Score: 0.684514888810536


for different depths

In [31]:
from sklearn.tree import DecisionTreeRegressor
depths = [3, 5, 7, 10, 15, 20, None]

for depth in depths:

    model = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            (
                'model',
                DecisionTreeRegressor(
                    max_depth=depth,
                    random_state=42
                )
            )
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    print(
        "Max Depth:", depth,
        "R²:", round(r2, 4)
    )

Max Depth: 3 R²: 0.6584
Max Depth: 5 R²: 0.7158
Max Depth: 7 R²: 0.7644
Max Depth: 10 R²: 0.8016
Max Depth: 15 R²: 0.7786
Max Depth: 20 R²: 0.7335
Max Depth: None R²: 0.6845


gridsearchcv

In [32]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            DecisionTreeRegressor(
                random_state=42
            )
        )
    ]
)

In [33]:
param_grid = {
    'model__max_depth': [5, 7, 10, 12, 15],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10]
}

In [34]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

In [35]:
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [5, 7, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added

In [36]:
print(grid_search.best_params_)

{'model__max_depth': 12, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}


In [37]:
best_dt = grid_search.best_estimator_

In [38]:
y_pred = best_dt.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(y_test, y_pred)

print("Best Decision Tree")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

Best Decision Tree
MAE: 15782.641197185856
RMSE: 21398.503208419035
R²: 0.8209905324747009


#Linear Regression

In [39]:
from sklearn.linear_model import LinearRegression
model_lr=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',LinearRegression())
    ]
)
model_lr.fit(X_train,y_train)
y_train_pred=model_lr.predict(X_train)
pred_lr=model_lr.predict(X_test)

In [40]:
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score,mean_squared_error
mae_lr = mean_absolute_error(y_test, pred_lr)

rmse_lr = np.sqrt(
    mean_squared_error(y_test, pred_lr)
)
r2_train=r2_score(y_train,y_train_pred)
r2_lr = r2_score(y_test, pred_lr)

print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print(r2_lr)
print("R2:", r2_lr)

MAE: 17016.128900822278
RMSE: 21962.631986371427
0.8114276656690196
R2: 0.8114276656690196


#Lasso Regression

In [41]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=10000))
])

param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search_lasso = GridSearchCV(
    lasso_pipeline,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search_lasso.fit(X_train, y_train)

print("Best Alpha:", grid_search_lasso.best_params_)
print("Best CV RMSE:", -grid_search_lasso.best_score_)

Best Alpha: {'model__alpha': 10}
Best CV RMSE: 21927.994310612637


In [42]:
best_ridge = grid_search_lasso.best_estimator_

y_pred = best_ridge.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Test MAE:", mae)
print("Test RMSE:", rmse)
print("Test R²:", r2)

Test MAE: 16975.104768535824
Test RMSE: 21934.316765077834
Test R²: 0.8119135842454437


#KNN

In [43]:
from sklearn.neighbors import KNeighborsRegressor
for k in [3, 5, 7, 9, 11, 15, 20]:

    model = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', KNeighborsRegressor(n_neighbors=k))
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    r2_scores=[]
    r2_scores.append(r2)

    print(k, r2)

3 0.42069747362120213
5 0.4643007182911021
7 0.4848164304164665
9 0.4833873477083165
11 0.47559632795623485
15 0.4448977542525684
20 0.409193733696367


In [44]:
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score,mean_squared_error
mae_kn = mean_absolute_error(y_test, y_pred)

rmse_kn = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2_kn = r2_score(y_test, y_pred)

print("MAE:", mae_kn)
print("RMSE:", rmse_kn)
print("R2:", r2_kn)

MAE: 30357.87837465565
RMSE: 38874.77982290459
R2: 0.409193733696367


#Support vector machine

In [45]:
from sklearn.svm import SVR
svr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVR())
])
svr_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['job_id','job_title','salary_currency',...,'job_description_length', 'benefits_score','company_name']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automaticall

In [46]:
y_pred = svr_model.predict(X_test)

In [47]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 40144.210518782726
RMSE: 51841.78750718203
R²: -0.05067680207987779
